# **Mount Drive**

# **Download SVI US Counties**

In [ ]:
# !wget https://svi.cdc.gov/Documents/Data/2000/db/states_counties/SVI_2000_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/
# !wget https://svi.cdc.gov/Documents/Data/2010/db/states_counties/SVI_2010_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/
# !wget https://svi.cdc.gov/Documents/Data/2014/db/states_counties/SVI_2014_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/
# !wget https://svi.cdc.gov/Documents/Data/2016/db/states_counties/SVI_2016_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/
# !wget https://svi.cdc.gov/Documents/Data/2018/db/states_counties/SVI_2018_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/
# !wget https://svi.cdc.gov/Documents/Data/2020/db/states_counties/SVI_2020_US_county.zip -P <DATA_ROOT>/SVI/SVI_zipFiles/

In [ ]:
import zipfile
from glob import glob
zip_path = '<DATA_ROOT>/SVI/SVI_zipFiles/*.zip'
files = glob(zip_path)


for zip_file in files:
  extract_dir = '<DATA_ROOT>/SVI/SVI_gdb/'
  with zipfile.ZipFile(zip_file, 'r') as zip_ref:
      zip_ref.extractall(extract_dir)


In [ ]:
import zipfile
zip_path = '<DATA_ROOT>/SVI/USA/USA.zip'
extract_dir = '<DATA_ROOT>/SVI/USA/'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# **Export SVI shp from GDB**

In [ ]:
!pip install geopandas
!pip install fiona
!pip install pyproj

In [ ]:
from glob import glob
zip_path = '<DATA_ROOT>/SVI/SVI_gdb/*.gdb'
files = sorted(glob(zip_path))
f2000 = files[0]
f2010 = files[1]
f2014_2020 = files[2:]
f2014_2020

In [ ]:
import geopandas as gpd
import fiona
import re

shp_path = '<DATA_ROOT>/SVI/SVI_shp/'

for gdb_file in f2014_2020:
  print(gdb_file)

  with fiona.open(gdb_file, 'r') as geodb:
      gdf = gpd.GeoDataFrame.from_features(geodb)

  gdf_cols = list(gdf.columns)
  state_county_cols = ['geometry', 'ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS' , 'LOCATION']
  for col in state_county_cols:
    if col in gdf_cols: continue
    state_county_cols.remove(col)

  r = re.compile("EPL_")
  epl_vars = list(filter(r.match, gdf_cols))
  print('num of EPL vars:',len(epl_vars))

  r = re.compile("RPL_")
  rpl_vars = list(filter(r.match, gdf_cols))
  print('num of RPL vars:', len(rpl_vars))

  SVI_vars = state_county_cols + rpl_vars + epl_vars
  gdf_new = gdf[SVI_vars]
  svi_fname = gdb_file.split('/')[-1].split('.')[0]
  gdf_new.to_file(shp_path+svi_fname)


In [ ]:
import geopandas as gpd
import fiona
import re

shp_path = '<DATA_ROOT>/SVI/SVI_shp/'

gdb_file = f2010

with fiona.open(gdb_file, 'r') as geodb:
    gdf = gpd.GeoDataFrame.from_features(geodb)

gdf_cols = list(gdf.columns)
state_county_cols = ['geometry', 'ST', 'STATE', 'FIPS', 'LOCATION']

for col in state_county_cols:
  if col in gdf_cols: continue
  state_county_cols.remove(col)

r = re.compile("E_PL_")
epl_vars0 = list(filter(r.match, gdf_cols))
epl_vars = [s.replace('E_PL_', 'EPL_') for s in epl_vars0]
epl_vars[epl_vars.index('EPL_NOHSDIP')] = 'EPL_NOHSDP'
print('num of EPL vars:',len(epl_vars))

r = re.compile("PL_")
pl_vars0 = list(filter(r.match, gdf_cols))
pl_vars = [s.replace('PL_', 'EPL_') for s in pl_vars0]
pl_vars[pl_vars.index('EPL_SNGPRNT')] = 'EPL_SNGPNT'
pl_vars[pl_vars.index('EPL_MINORITY')] = 'EPL_MINRTY'
print('num of PL vars:',len(pl_vars))

r = re.compile("R_PL_")
rpl_vars0 = list(filter(r.match, gdf_cols))
rpl_vars = [s.replace('R_PL_', 'RPL_') for s in rpl_vars0]
print(rpl_vars)
print('num of RPL vars:', len(rpl_vars))

SVI_vars_df = state_county_cols + rpl_vars0 + epl_vars0 + pl_vars0
SVI_vars = state_county_cols + rpl_vars + epl_vars + pl_vars
gdf_new = gdf[SVI_vars_df]
gdf_new.columns = SVI_vars
svi_fname = gdb_file.split('/')[-1].split('.')[0]
gdf_new.to_file(shp_path+svi_fname)


In [ ]:
import geopandas as gpd
import fiona
import re

shp_path = '<DATA_ROOT>/SVI/SVI_shp/'

gdb_file = f2000

with fiona.open(gdb_file, 'r') as geodb:
    gdf = gpd.GeoDataFrame.from_features(geodb)

gdf_cols = list(gdf.columns)
gdf_cols
state_county_cols = ['geometry', 'STATE_FIPS', 'CNTY_FIPS', 'STCOFIPS', 'STATE_NAME', 'STATE_ABBR', 'COUNTY']


cols_map = {
  'USG1V1P':'EPL_POV',
  'USG1V2P':'EPL_UNEMP',
  'USG1V3P':'EPL_PCI',
  'USG1V4P':'EPL_NOHSDP',
  'USG1TP':'RPL_THEME1',
  'USG2V1P':'EPL_AGE65',
  'USG2V2P':'EPL_AGE17',
  'USG2V3P':'EPL_DISABL',
  'USG2V4P':'EPL_SNGPNT',
  'USG2TP':'RPL_THEME2',
  'USG3V1P':'EPL_MINRTY',
  'USG3V2P':'EPL_LIMENG',
  'USG3TP':'RPL_THEME3',
  'USG4V1P':'EPL_MUNIT',
  'USG4V2P':'EPL_MOBILE',
  'USG4V3P':'EPL_CROWD',
  'USG4V4P':'EPL_NOVEH',
  'USG4V5P':'EPL_GROUPQ',
  'USG4TP':'RPL_THEME4',
  'USTP':'RPL_THEMES'
}

gdf_new = gdf[state_county_cols + list(cols_map.keys())]
gdf_new.rename(columns=cols_map, inplace=True)

svi_fname = gdb_file.split('/')[-1].split('.')[0]
gdf_new.to_file(shp_path+svi_fname)

# **Plot SVI shp**

In [ ]:
import geopandas as gpd
from glob import glob

USA_dir = '<DATA_ROOT>/SVI/USA/'
US_gdf = gpd.read_file(USA_dir)
bbox = US_gdf.bounds

shp_path = '<DATA_ROOT>/SVI/SVI_shp/*'
dirs = sorted(glob(shp_path))

In [ ]:
import matplotlib.pyplot as plt
i = 0
fig, axes = plt.subplots(3, 2, figsize=(20, 10))
for shp in dirs:
  gdf = gpd.read_file(shp)
  gdf.crs = "EPSG:4326"
  clipped_gdf = gdf.cx[bbox.minx:bbox.maxx, bbox.miny:bbox.maxy]
  classification_column = "RPL_THEMES"
  clipped_gdf = clipped_gdf.loc[clipped_gdf[classification_column]>0]
  clipped_gdf.plot(ax=axes[i//2,i%2],column=classification_column, legend=True, cmap="viridis")
  axes[i//2,i%2].set_title(shp.split('/')[-1] + ' ' + classification_column,fontsize=14)
  i = i + 1

plt.tight_layout()
plt.show()

